# 03 · Ingestão Bronze via Auto Loader

Lê os arquivos Parquet do diretório de landing
(`/Volumes/antecipeai/landing/raw/incidentes/`) via **Auto Loader**
(`cloudFiles`), que faz leitura incremental — só processa arquivos novos
desde a última execução, controlado por checkpoint.

**Bronze = "na íntegra"**: nenhum tratamento de tipo, nenhuma limpeza.
Os dados já chegam como string (decisão tomada no notebook `02`), e aqui
só adicionamos metadata técnica de ingestão:
- `_ingested_at`: timestamp de quando a linha entrou na Bronze
- `_source_file`: arquivo de origem de cada linha (rastreabilidade)

Usamos `.trigger(availableNow=True)`: processa tudo que estiver disponível
agora e para — sem manter um cluster rodando 24/7 (importante pro custo
zero do Databricks Free). Quando novos arquivos chegarem na landing,
basta rodar este notebook de novo.

In [ ]:
%run ./00_config

## Paths de checkpoint, schema location e origem/destino

In [ ]:
from pyspark.sql import functions as F

source_path = volume_path("incidentes")
checkpoint_path = volume_path("_checkpoints/bronze_incidentes")
schema_location = volume_path("_schema/bronze_incidentes")
bronze_table = qualified_table(SCHEMA_BRONZE, "incidentes")

print("Origem (landing):     ", source_path)
print("Checkpoint:            ", checkpoint_path)
print("Schema location:       ", schema_location)
print("Tabela destino (Bronze):", bronze_table)

## Column Mapping — obrigatório antes de criar a tabela

Os nomes de coluna vêm exatamente como no arquivo de origem (`Grupo
designado`, `Descrição resumida`, `Entrou para KPI?`...), com espaços e
caracteres que o Delta rejeita por padrão
(`DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES`). Para manter os nomes
originais "na íntegra" — como a Bronze exige — sem sanitizar/renomear
nessa camada (isso fica pra Silver), precisamos habilitar Column
Mapping **antes** da tabela ser criada pela primeira vez.

**Isso não funciona como `.option()` no `writeStream`** — essas opções
só afetam o comportamento do stream em si, não as propriedades da
tabela Delta criada por trás. O jeito documentado pela Databricks é
setar isso como **default de sessão**: qualquer tabela Delta nova criada
nesta sessão (via `CREATE TABLE`, `saveAsTable` ou `toTable`) já nasce
com essas propriedades, sem precisar declarar schema/DDL na mão — o que
preservaria o objetivo de schema-on-read do Auto Loader.

In [ ]:
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

print("Defaults de sessão para Column Mapping aplicados — válidos para qualquer tabela Delta nova criada a partir daqui.")

## Reset opcional da tabela Bronze

Se a tabela `bronze.incidentes` já existe de uma tentativa anterior
**sem** Column Mapping habilitado, o Delta ignora os defaults de sessão
(eles só valem na criação). Este widget existe pra isso: fica desligado
por padrão (não apaga nada sozinho ao rodar o notebook inteiro de novo),
e só derruba a tabela + limpa o checkpoint/schema location do Auto
Loader quando você explicitamente muda pra `true`.

In [ ]:
dbutils.widgets.dropdown("reset_bronze_table", "false", ["false", "true"], "Derrubar bronze.incidentes e recriar do zero?")

if dbutils.widgets.get("reset_bronze_table") == "true":
    spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
    dbutils.fs.rm(checkpoint_path, recurse=True)
    dbutils.fs.rm(schema_location, recurse=True)
    print(f"Reset concluído: {bronze_table} derrubada, checkpoint e schema location limpos.")
else:
    print("Reset não solicitado (reset_bronze_table=false) — nada foi apagado.")

## Leitura incremental com Auto Loader

`cloudFiles.inferColumnTypes` fica `false` de propósito — mesmo o
Parquet de origem já vindo 100% string (garantido no notebook `02`),
deixamos explícito aqui que a Bronze nunca deve inferir tipos. Isso é o
que torna essa camada resiliente a mudanças de schema na origem: uma
coluna nova na próxima extração da Locaweb é automaticamente capturada
(schema evolution do Auto Loader), sem quebrar o job.

In [ ]:
raw_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

bronze_stream = (
    raw_stream
    .withColumn("_ingested_at", F.current_timestamp())
    # _metadata.file_path (coluna oculta do Spark, exposta automaticamente em
    # toda leitura de arquivo) — NÃO usar F.input_file_name(): o Unity
    # Catalog bloqueia essa função legada
    # ([UC_COMMAND_NOT_SUPPORTED.WITH_RECOMMENDATION]), já que a linhagem de
    # arquivo passa a ser controlada pelo próprio UC. É Spark padrão desde a
    # 3.4 (SPARK-37273), não é exclusividade Databricks.
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

## Escrita incremental na tabela Delta da Bronze

In [ ]:
query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print(f"Ingestão concluída. Linhas na Bronze agora: {spark.table(bronze_table).count()}")

## Conferência

In [ ]:
df_bronze = spark.table(bronze_table)
df_bronze.printSchema()
display(df_bronze.limit(10))

In [ ]:
# Resultado esperado (validado com o dataset real fora do Databricks antes de
# escrever este notebook): 122.543 linhas, 19 colunas originais + as 2 colunas
# de metadata (_ingested_at, _source_file) = 21 colunas. Todas as 19 colunas
# originais devem aparecer como StringType — se alguma vier tipada diferente,
# a etapa de conversão do notebook 02 não rodou como esperado.